In [0]:
df_nov = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv", header=True, inferSchema=True)

In [0]:
dbutils.fs.ls("/Volumes/workspace/ecommerce/ecommerce_data/")


[FileInfo(path='dbfs:/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv', name='2019-Nov.csv', size=9006762395, modificationTime=1767958694000),
 FileInfo(path='dbfs:/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv', name='2019-Oct.csv', size=5668612855, modificationTime=1767958830000),
 FileInfo(path='dbfs:/Volumes/workspace/ecommerce/ecommerce_data/delta/', name='delta/', size=0, modificationTime=1768304361338),
 FileInfo(path='dbfs:/Volumes/workspace/ecommerce/ecommerce_data/df_nov/', name='df_nov/', size=0, modificationTime=1768304361338),
 FileInfo(path='dbfs:/Volumes/workspace/ecommerce/ecommerce_data/parquet/', name='parquet/', size=0, modificationTime=1768304361338)]

In [0]:
updates = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
    header=True,
    inferSchema=True
).limit(1000)



In [0]:
from delta.tables import DeltaTable
# Load target Delta table
deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/ecommerce/ecommerce_data/delta/df_nov")
# Load incremental updates
updates = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",
    header=True,
    inferSchema=True)
# Perform MERGE
deltaTable.alias("t").merge(
    updates.alias("s"),
    "t.user_session = s.user_session AND t.event_time = s.event_time AND t.product_id = s.product_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(deltaTable.history())

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-01-13T11:42:28.000Z,71962889837098,niharikaganji22@gmail.com,MERGE,"Map(predicate -> [""(((user_session#13284 = user_session#13310) AND (event_time#13276 = event_time#13302)) AND (product_id#13278 = product_id#13304))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1962836806931815),0113-112931-wvjggf8r-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 43, numTargetBytesAdded -> 1405244778, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 63249, materializeSourceTimeMs -> 29884, numTargetRowsInserted -> 42448764, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 19611, numTargetRowsUpdated -> 0, numOutputRows -> 42448764, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 42448764, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 13622)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
2,2026-01-13T10:27:23.000Z,71962889837098,niharikaganji22@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1962836806931815),0113-095114-jdjw2auf-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 206824475, p25FileSize -> 3447988, numDeletionVectorsRemoved -> 6, minFileSize -> 3447988, numAddedFiles -> 1, maxFileSize -> 3447988, p75FileSize -> 3447988, p50FileSize -> 3447988, numAddedBytes -> 3447988)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2026-01-13T10:27:17.000Z,71962889837098,niharikaganji22@gmail.com,MERGE,"Map(predicate -> [""(((user_session#13942 = user_session#13968) AND (event_time#13934 = event_time#13960)) AND (product_id#13936 = product_id#13962))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1962836806931815),0113-095114-jdjw2auf-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 73, numTargetBytesAdded -> 2220767422, numTargetBytesRemoved -> 1314978219, numTargetDeletionVectorsAdded -> 6, numTargetRowsMatchedUpdated -> 41130527, executionTimeMs -> 92408, materializeSourceTimeMs -> 44506, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 23342, numTargetRowsUpdated -> 41130527, numOutputRows -> 41130533, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41019511, numTargetFilesRemoved -> 36, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 24410)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2026-01-12T14:18:18.000Z,71962889837098,niharikaganji22@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3142724936969334),0112-141113-6ktz60x9-v2n,null,WriteSerializable,false,"Map(numFiles -> 68, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 67501979, numOutputBytes -> 2451735159)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
# Time travel
v0 = spark.read.format("delta").option("versionAsOf", 0).load("/delta/df_nov")
yesterday = spark.read.format("delta") \
    .option("timestampAsOf", "2024-01-01").load("/delta/df_nov")

In [0]:
# Optimize
spark.sql("OPTIMIZE df_nov_table ZORDER BY (event_type, user_id)")
spark.sql("VACUUM df_nov_table RETAIN 168 HOURS")

DataFrame[path: string]

In [0]:
deltaTable.history().display(truncate= True)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-01-13T10:27:23.000Z,71962889837098,niharikaganji22@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1962836806931815),0113-095114-jdjw2auf-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 206824475, p25FileSize -> 3447988, numDeletionVectorsRemoved -> 6, minFileSize -> 3447988, numAddedFiles -> 1, maxFileSize -> 3447988, p75FileSize -> 3447988, p50FileSize -> 3447988, numAddedBytes -> 3447988)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2026-01-13T10:27:17.000Z,71962889837098,niharikaganji22@gmail.com,MERGE,"Map(predicate -> [""(((user_session#13942 = user_session#13968) AND (event_time#13934 = event_time#13960)) AND (product_id#13936 = product_id#13962))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1962836806931815),0113-095114-jdjw2auf-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 73, numTargetBytesAdded -> 2220767422, numTargetBytesRemoved -> 1314978219, numTargetDeletionVectorsAdded -> 6, numTargetRowsMatchedUpdated -> 41130527, executionTimeMs -> 92408, materializeSourceTimeMs -> 44506, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 23342, numTargetRowsUpdated -> 41130527, numOutputRows -> 41130533, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41019511, numTargetFilesRemoved -> 36, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 24410)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2026-01-12T14:18:18.000Z,71962889837098,niharikaganji22@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3142724936969334),0112-141113-6ktz60x9-v2n,null,WriteSerializable,false,"Map(numFiles -> 68, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 67501979, numOutputBytes -> 2451735159)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
#choose specific version
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load('/Volumes/workspace/ecommerce/ecommerce_data/delta/df_nov')
df_v0.display()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-11-01T00:00:00.000Z,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
2019-11-01T00:00:00.000Z,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2019-11-01T00:00:01.000Z,view,17302664,2053013553853497655,null,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
2019-11-01T00:00:01.000Z,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
2019-11-01T00:00:01.000Z,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2
2019-11-01T00:00:01.000Z,view,1306894,2053013558920217191,computers.notebook,hp,360.09,520772685,816a59f3-f5ae-4ccd-9b23-82aa8c23d33c
2019-11-01T00:00:01.000Z,view,1306421,2053013558920217191,computers.notebook,hp,514.56,514028527,df8184cc-3694-4549-8c8c-6b5171877376
2019-11-01T00:00:02.000Z,view,15900065,2053013558190408249,null,rondell,30.86,518574284,5e6ef132-4d7c-4730-8c7f-85aa4082588f
2019-11-01T00:00:02.000Z,view,12708937,2053013553559896355,null,michelin,72.72,532364121,0a899268-31eb-46de-898d-09b2da950b24
2019-11-01T00:00:02.000Z,view,1004258,2053013555631882655,electronics.smartphone,apple,732.07,532647354,d2d3d2c6-631d-489e-9fb5-06f340b85be0
